In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
### 所需要的所有文件：
# 物流的发货清单
# 财务的发货清单
# 核算价
# PLM的生命周期全表
### 输出的所有文件
# 统计周期内产品核算价汇总 用于物料精简报告，因为里面有国内国外
# 合并物流-财务-产品组-核算价-国内-用于低效-长尾 ，用于低效-长尾报告，里面只有国内
# 单型号贡献-统计值



### MAP关系汇总1、渠道对照 2、最终表格产品类别对应的产品组集合 3、产品组集合

In [6]:
# 物流的渠道对照关系清洗用
month = 202512
Channel_map = {
'工程': '工程',
'零售':'零售',
'电商不可售':'电商',
'电商':'电商',
'内部处理通用':'电商',
'借出渠道':'非零售工程电商',
'新品':'电商',
'战略电商':'电商',
'转出渠道':'非零售工程电商',
'每誉':'每誉',
'商净':'商净',
'渠道':'非零售工程电商',
'海外':'海外',
'调出渠道':'非零售工程电商',
'非零售工程电商':'非零售工程电商',
'米博新零售':'米博新零售'
}
#用于合并计算
productgroupset_map = {
    '吸油烟机':['吸油烟机'],
    '灶具':['灶具'],
    '烤箱':['烤箱'],
    '蒸箱':['蒸箱'],
    '微波炉':['微波炉'],
    '蒸烤烹饪机':['蒸烤烹饪机'],
    '蒸烤微烹饪机':['蒸烤微烹饪机'],
    '蒸微':['蒸微'],
    '蒸烤微合计':['烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微'],
    '灶消烹饪机':['灶消烹饪机'],
    '灶蒸烹饪机':['灶蒸烹饪机'],
    '灶蒸烤烹饪机':['灶蒸烤烹饪机'],
    '灶烤烹饪机':['灶烤烹饪机'],
    '灶集成':['灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '烹饪产品线合计':['灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '消毒柜':['消毒柜'],
    '热水器':['热水器'],
    '两用炉':['两用炉'],
    '热水器两用炉合计':['热水器','两用炉'],
    '家用净水机':['家用净水机'],
    '商用净水机':['商用净水机'],
    '净热产品线合计':['热水器','两用炉','家用净水机','商用净水机'],
    '水槽洗碗机':['水槽洗碗机'],
    '嵌入式洗碗机':['嵌入式洗碗机'],
    '洗碗机产品线合计':['水槽洗碗机','嵌入式洗碗机'],
    '国内合计':['吸油烟机','灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机','消毒柜','热水器','两用炉','家用净水机','商用净水机','水槽洗碗机','嵌入式洗碗机'],
}
# 统计的产品组
productgroup_list = ['吸油烟机','灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机','消毒柜','热水器','两用炉','热水器两用炉合计','家用净水机','商用净水机','净热产品线合计','水槽洗碗机','嵌入式洗碗机','洗碗机产品线合计','国内合计']


### 获取物流发货数据

In [7]:
#将这些excel的表格数据进行上下拼接，只要字段：商品编码、商品名称、渠道、实际出库数量
folder_path = fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\物流发货"  # 请替换为实际的文件夹路径
excel_files = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
        sheet_names = file[5:].replace(file[-5:],'')
        if (file.endswith('.xlsx') or file.endswith('.xls')) and not file.startswith('~$'):
            file_path = os.path.join(root, file)
            excel_files.append((file, sheet_names, file_path))
print(excel_files)

[('2025年10月明细.xlsx', '10月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年10月明细.xlsx'), ('2025年11月明细.xlsx', '11月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年11月明细.xlsx'), ('2025年12月明细.xlsx', '12月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年12月明细.xlsx'), ('2025年1月明细.xlsx', '1月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年1月明细.xlsx'), ('2025年2月明细.xlsx', '2月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年2月明细.xlsx'), ('2025年3月明细.xlsx', '3月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年3月明细.xlsx'), ('2025年4月明细.xlsx', '4月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年4月明细.xlsx'), ('2025年5月明细.xlsx', '5月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年5月明细.xlsx'), ('2025年6月明细.xlsx', '6月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年6月明细.xlsx'), ('2025年7月明细.xlsx', '7月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年7月明细.xlsx'), ('2025年8月明细.xlsx', '8月明细', 'D:\\000物料报表\\202512\\单型号贡献-低效-长尾\\物流发货\\2025年8月明细.xlsx'), ('2025年9月明细.xlsx', '9月明细', 'D:\\000物料报表\\202

In [13]:
#如果有报错请提示报错信息
df_dcs = pd.DataFrame()
for file in excel_files:
    try:
        df_temp = pd.read_excel(file[2], sheet_name=file[1])
        print(f'成功读取{file[0]}的{file[1]}表')
    except:
        print(f'读取{file[0]}的{file[1]}表失败')
    if '实际总数量' in df_temp.columns:
        df_temp = df_temp.rename(columns={'实际总数量':'实际出库数量'})
    df_temp['发货月份'] = file[0][:8]
    df_temp = df_temp[['商品编码', '渠道', '实际出库数量','发货月份','发货仓','收货仓']]
    df_dcs = pd.concat([df_dcs, df_temp], axis=0).reset_index(drop=True)
df_dcs = df_dcs.dropna(how='all').reset_index(drop=True)  # 仅当一行所有值都是NaN时才删除
df_dcs['商品编码'] = df_dcs['商品编码'].astype(str)
df_dcs['渠道'].value_counts()


成功读取2025年10月明细.xlsx的10月明细表
成功读取2025年11月明细.xlsx的11月明细表
成功读取2025年12月明细.xlsx的12月明细表
成功读取2025年1月明细.xlsx的1月明细表
成功读取2025年2月明细.xlsx的2月明细表
成功读取2025年3月明细.xlsx的3月明细表
成功读取2025年4月明细.xlsx的4月明细表
成功读取2025年5月明细.xlsx的5月明细表
成功读取2025年6月明细.xlsx的6月明细表
成功读取2025年7月明细.xlsx的7月明细表
成功读取2025年8月明细.xlsx的8月明细表
成功读取2025年9月明细.xlsx的9月明细表


渠道
零售        489375
工程         21166
电商         10507
电商不可售       6616
海外          2359
每誉           188
战略电商         129
新品            67
内部处理通用        15
商净            14
借出渠道          14
渠道             4
调出渠道           2
转出渠道           2
米博新零售          1
Name: count, dtype: int64

### 进行渠道清洗

In [9]:
df0 = df_dcs.copy()
df0['物料编码'] = df0['商品编码'].astype(str)
df0['渠道'] = df0['渠道'].map(Channel_map).fillna('非零售工程电商')
df0 = df0[(df0['渠道']!='无') & 
        (df0['物料编码'].str.len()>=10)
        ].reset_index(drop=True)
df0['实际出库数量'] = df0['实际出库数量'].astype(float)
print(df0['渠道'].value_counts())
df0
# 

渠道
零售       489375
工程        21166
电商        17334
海外         2359
每誉          188
商净           14
米博新零售         1
Name: count, dtype: int64


,商品编码,渠道,实际出库数量,发货月份,物料编码
0,1001001500116,零售,12.0,2025年10月,1001001500116
1,1009000600033,零售,1.0,2025年10月,1009000600033
2,1009000500035,零售,3.0,2025年10月,1009000500035
3,1001001500131,零售,6.0,2025年10月,1001001500131
4,1002003700049,零售,5.0,2025年10月,1002003700049
...,...,...,...,...,...
530432,1018000500026,海外,4.0,2025年9月明,1018000500026
530433,1001000500212,海外,2.0,2025年9月明,1001000500212
530434,1001000500390,海外,10.0,2025年9月明,1001000500390
530435,1001000500390,海外,1.0,2025年9月明,1001000500390


In [19]:
# df0.to_excel(r'C:\Users\zhangbon\Desktop\发货数据.xlsx', index=False)

### 将产品组信息、核算价匹配进去

In [10]:
# PLM产品数据导入
df_plm = pd.read_excel(fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\产品生命周期状态全表.xlsx")

#数据清洗
df_plm = df_plm[df_plm['物料号'].str.len()>10]
df_plm['物料编码'] = df_plm['物料号'].apply(lambda x: x[:13])
df_plm[['物料编码','产品组','标准型号','国内/海外','产品状态']] = df_plm[['物料编码','产品组','标准型号','国内/海外','产品状态']].astype(str)
df_plm['开始销售时间'] = pd.to_datetime(df_plm['开始销售时间'],format='mixed')
df_plm['停止销售时间'] = pd.to_datetime(df_plm['停止销售时间'],format='mixed')

# 产品维度的数据导入
df_plm1 = df_plm[['物料编码','产品型号','产品组','标准型号','国内/海外','产品状态']].drop_duplicates()

# 产品+渠道的数据导入
df_plm['渠道'] = df_plm['下属渠道']
df_plm2 = df_plm[['物料编码','渠道','对应渠道状态','开始销售时间','停止销售时间']].drop_duplicates()
# df_product_group

e:\python\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [11]:
# 导入财务的核算价
df_price = pd.read_excel(fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\核算价.xlsx")
df_price['产品编码'] = df_price['产品编码'].astype(str)
df_price['系统核算价'] = df_price['系统核算价'].astype(float)
# df_price
product_price_map = dict(zip(df_price['产品编码'],df_price['系统核算价']))

In [12]:

df1 = df0.copy()
df1 = pd.merge(df1, df_plm1, on='物料编码', how='left')
df1 = pd.merge(df1, df_plm2, on=['物料编码','渠道'], how='left')
df1['系统核算价'] = df1['商品编码'].map(product_price_map)
df1['核算价'] = df1['系统核算价'] * df1['实际出库数量']
df1 = df1[df1['商品编码'].str.startswith('10')].reset_index(drop=True)
#这里要提示哪些数据的系统核算价是空的
print(f'国内没有系统核算价的是这些数据\n{df1[(df1['系统核算价'].isnull()&(df1['国内/海外']=='国内'))]['商品编码'].drop_duplicates()}')
print(len(df1))
df1

国内没有系统核算价的是这些数据
Series([], Name: 商品编码, dtype: object)
524227


,商品编码,渠道,实际出库数量,发货月份,物料编码,产品型号,产品组,标准型号,国内/海外,产品状态,对应渠道状态,开始销售时间,停止销售时间,系统核算价,核算价
0,1001001500116,零售,12.0,2025年10月,1001001500116,CXW-358-Z8T(不带罩),吸油烟机,Z8T,国内,停止销售,停止销售,2022-12-10 12:00:00,2025-11-24 16:51:46,3358.0,40296.0
1,1009000600033,零售,1.0,2025年10月,1009000600033,ZK50-02-F1,蒸烤烹饪机,ZK50-02-F1,国内,量产,在售,2025-06-09 12:00:00,NaT,3450.0,3450.0
2,1009000500035,零售,3.0,2025年10月,1009000500035,JZT-ZK46-X2,灶蒸烤烹饪机,JZT-ZK46-X2,国内,量产,在售,2025-04-29 12:00:00,NaT,5280.0,15840.0
3,1001001500131,零售,6.0,2025年10月,1001001500131,CXW-358-02-Z6TA(不带罩),吸油烟机,02-Z6TA,国内,停止销售,停止销售,2024-04-02 12:00:00,2025-11-24 16:51:46,2988.0,17928.0
4,1002003700049,零售,5.0,2025年10月,1002003700049,JZT-01-H8B-12T,灶具,H8B,国内,量产,在售,2024-01-29 12:00:00,NaT,2550.0,12750.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
524222,1018000500026,海外,4.0,2025年9月明,1018000500026,BD2B-G2-Y-USFA,嵌入式洗碗机,nan,海外,小批量,未售,NaT,NaT,NaN,NaN
524223,1001000500212,海外,2.0,2025年9月明,1001000500212,JQG7501-USFA,吸油烟机,nan,海外,量产,在售,2022-12-30 12:00:00,NaT,NaN,NaN
524224,1001000500390,海外,10.0,2025年9月明,1001000500390,JQG9006-W-USFA,吸油烟机,nan,海外,小批量,未售,NaT,NaT,NaN,NaN
524225,1001000500390,海外,1.0,2025年9月明,1001000500390,JQG9006-W-USFA,吸油烟机,nan,海外,小批量,未售,NaT,NaT,NaN,NaN


### 这里导出的数据，包含了国内和海外的整机,这个用于物料精简（零件分析）报告的分析，用于计算收入

In [23]:
# 物流和财务的发货收入中是包含了1012开头的样机的，PLM中是只有成品的，所以这里样机的产品组是空的，样机不在我们报告的统计范围内
# 这份导出的数据中是包含了国内国外的整机以及样机等数据，（样机等不会有产品组信息，因为PLM产品生命周期表中是没有样机的）
df1.to_excel(fr'D:\000物料报表\{month}\单物料产值\统计周期内产品核算价汇总.xlsx',index=False)


### 保留范围内的产品组(这样是会只剩下整机)、国内产品、停止发货以前（不含停止发货）、三大渠道，这里要注意补充核算价！！！！！（在这里会输出一个文件用于后续低效和长尾的分析）

In [24]:
df2 = df1.copy()
df2 = df2[(df2['国内/海外']=='国内') & 
                (df2['产品状态'].isin(['开发','停止生产','量产','停止销售','小批量','样机','退市预警','创建',])) &
                (df2['渠道'].isin(['零售','工程','电商']))&
                (df2['产品组'].isin(productgroup_list))
                ].reset_index(drop=True)
#这里要提示哪些数据的系统核算价是空的
print(f'没有系统核算价的是这些数据\n{df2[df2['系统核算价'].isnull()]['商品编码'].drop_duplicates()}')
print(len(df2))
# 这里导出数据用于低效和长尾的分析。因为低效和长尾只看国内的成品情况（不包含样机，且在我们的统计产品组范围内）
df2.to_excel(fr'D:\000物料报表\{month}\单型号贡献-低效-长尾\合并物流-财务-产品组-核算价-国内-用于低效-长尾.xlsx',index=False)
df2


没有系统核算价的是这些数据
Series([], Name: 商品编码, dtype: object)
497619


,商品编码,渠道,实际出库数量,物料编码,产品型号,产品组,标准型号,国内/海外,产品状态,对应渠道状态,开始销售时间,停止销售时间,系统核算价,核算价
0,1001001500116,零售,12.0,1001001500116,CXW-358-Z8T(不带罩),吸油烟机,Z8T,国内,停止销售,停止销售,2022-12-10 12:00:00,2025-11-24 16:51:46,3358.0,40296.0
1,1009000600033,零售,1.0,1009000600033,ZK50-02-F1,蒸烤烹饪机,ZK50-02-F1,国内,量产,在售,2025-06-09 12:00:00,NaT,3450.0,3450.0
2,1009000500035,零售,3.0,1009000500035,JZT-ZK46-X2,灶蒸烤烹饪机,JZT-ZK46-X2,国内,量产,在售,2025-04-29 12:00:00,NaT,5280.0,15840.0
3,1001001500131,零售,6.0,1001001500131,CXW-358-02-Z6TA(不带罩),吸油烟机,02-Z6TA,国内,停止销售,停止销售,2024-04-02 12:00:00,2025-11-24 16:51:46,2988.0,17928.0
4,1002003700049,零售,5.0,1002003700049,JZT-01-H8B-12T,灶具,H8B,国内,量产,在售,2024-01-29 12:00:00,NaT,2550.0,12750.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497614,1018001100016,零售,-12.0,1018001100016,JBCD7E-04-Z5,嵌入式洗碗机,JBCD7E-04-Z5,国内,量产,在售,2025-06-19 12:00:00,NaT,3750.0,-45000.0
497615,1018001100017,电商,6.0,1018001100017,JBCD7E-04-Y1,嵌入式洗碗机,JBCD7E-04-Y1,国内,量产,NaN,NaT,NaT,4500.0,27000.0
497616,1018001100017,零售,-6.0,1018001100017,JBCD7E-04-Y1,嵌入式洗碗机,JBCD7E-04-Y1,国内,量产,在售,2025-08-22 12:00:00,NaT,4500.0,-27000.0
497617,1018001100018,电商,6.0,1018001100018,JBCD7E-04-M-Y1,嵌入式洗碗机,JBCD7E-04-M-Y1,国内,量产,NaN,NaT,NaT,4500.0,27000.0


### 构建最终的统计结果表

In [25]:
df_calu = pd.DataFrame()
df_calu['产品类别'] = productgroupset_map.keys()
for k,v in productgroupset_map.items():
    # 零售渠道的统计
    vals = df2[(df2['产品组'].isin(v))&(df2['渠道'] == '零售')]['标准型号'].nunique()
    df_calu.loc[df_calu['产品类别']==k,'零售渠道标准型号数'] = vals
    vals = df2[(df2['产品组'].isin(v))&(df2['渠道'] == '零售')]['核算价'].sum()
    df_calu.loc[df_calu['产品类别']==k,'零售渠道核算价'] = vals

    # 工程渠道的统计
    vals = df2[(df2['产品组'].isin(v))&(df2['渠道'] == '工程')]['标准型号'].nunique()
    df_calu.loc[df_calu['产品类别']==k,'工程渠道标准型号数'] = vals
    vals = df2[(df2['产品组'].isin(v))&(df2['渠道'] == '工程')]['核算价'].sum()
    df_calu.loc[df_calu['产品类别']==k,'工程渠道核算价'] = vals

    # 电商渠道的统计
    vals = df2[(df2['产品组'].isin(v))&(df2['渠道'] == '电商')]['标准型号'].nunique()
    df_calu.loc[df_calu['产品类别']==k,'电商渠道标准型号数'] = vals
    vals = df2[(df2['产品组'].isin(v))&(df2['渠道'] == '电商')]['核算价'].sum()
    df_calu.loc[df_calu['产品类别']==k,'电商渠道核算价'] = vals

    # 合计
    vals = df2[(df2['产品组'].isin(v))]['标准型号'].nunique()
    df_calu.loc[df_calu['产品类别']==k,'汇总标准型号数'] = vals
    vals = df2[(df2['产品组'].isin(v))]['核算价'].sum()
    df_calu.loc[df_calu['产品类别']==k,'汇总核算价'] = vals
df_calu

,产品类别,零售渠道标准型号数,零售渠道核算价,工程渠道标准型号数,工程渠道核算价,电商渠道标准型号数,电商渠道核算价,汇总标准型号数,汇总核算价
0,吸油烟机,92.0,4.085039e+09,70.0,7.889209e+08,108.0,2.140111e+09,160.0,7.014071e+09
1,灶具,62.0,1.840085e+09,60.0,3.938705e+08,71.0,1.159758e+09,97.0,3.393714e+09
2,烤箱,6.0,5.023282e+07,5.0,4.220300e+06,5.0,4.261950e+06,7.0,5.871507e+07
3,蒸箱,5.0,6.009674e+07,4.0,2.158600e+06,4.0,1.148585e+07,6.0,7.374119e+07
4,微波炉,3.0,1.115407e+07,2.0,1.154960e+06,2.0,3.875450e+06,4.0,1.618448e+07
5,蒸烤烹饪机,25.0,6.156148e+08,20.0,6.428179e+07,32.0,1.238160e+08,37.0,8.037126e+08
6,蒸烤微烹饪机,5.0,2.255744e+07,4.0,1.574650e+06,6.0,5.599421e+07,6.0,8.012630e+07
7,蒸微,0.0,0.000000e+00,0.0,0.000000e+00,0.0,0.000000e+00,0.0,0.000000e+00
8,蒸烤微合计,44.0,7.596559e+08,35.0,7.339030e+07,49.0,1.994335e+08,60.0,1.032480e+09
9,灶消烹饪机,3.0,3.220889e+07,1.0,7.540000e+03,2.0,5.883800e+05,3.0,3.280481e+07


#### 输出结果

In [26]:
df_calu[['零售渠道核算价','电商渠道核算价','工程渠道核算价','汇总核算价']] = df_calu[['零售渠道核算价','电商渠道核算价','工程渠道核算价','汇总核算价']]/10000
df_calu['零售单型号贡献值'] = df_calu['零售渠道核算价']/df_calu['零售渠道标准型号数']
df_calu['工程单型号贡献值'] = df_calu['工程渠道核算价']/df_calu['工程渠道标准型号数']
df_calu['电商单型号贡献值'] = df_calu['电商渠道核算价']/df_calu['电商渠道标准型号数']
df_calu['汇总单型号贡献值'] = df_calu['汇总核算价']/df_calu['汇总标准型号数']
df_calu.to_excel(fr'D:\000物料报表\{month}\单型号贡献-低效-长尾\单型号贡献-统计值.xlsx',index=False)
df_calu


,产品类别,零售渠道标准型号数,零售渠道核算价,工程渠道标准型号数,工程渠道核算价,电商渠道标准型号数,电商渠道核算价,汇总标准型号数,汇总核算价,零售单型号贡献值,工程单型号贡献值,电商单型号贡献值,汇总单型号贡献值
0,吸油烟机,92.0,408503.9222,70.0,78892.0886,108.0,214011.1280,160.0,7.014071e+05,4440.260024,1127.029837,1981.584519,4383.794617
1,灶具,62.0,184008.4718,60.0,39387.0496,71.0,115975.8286,97.0,3.393713e+05,2967.878577,656.450827,1633.462375,3498.673711
2,烤箱,6.0,5023.2820,5.0,422.0300,5.0,426.1950,7.0,5.871507e+03,837.213667,84.406000,85.239000,838.786714
3,蒸箱,5.0,6009.6740,4.0,215.8600,4.0,1148.5850,6.0,7.374119e+03,1201.934800,53.965000,287.146250,1229.019833
4,微波炉,3.0,1115.4070,2.0,115.4960,2.0,387.5450,4.0,1.618448e+03,371.802333,57.748000,193.772500,404.612000
5,蒸烤烹饪机,25.0,61561.4833,20.0,6428.1791,32.0,12381.6015,37.0,8.037126e+04,2462.459332,321.408955,386.925047,2172.196322
6,蒸烤微烹饪机,5.0,2255.7440,4.0,157.4650,6.0,5599.4210,6.0,8.012630e+03,451.148800,39.366250,933.236833,1335.438333
7,蒸微,0.0,0.0000,0.0,0.0000,0.0,0.0000,0.0,0.000000e+00,NaN,NaN,NaN,NaN
8,蒸烤微合计,44.0,75965.5903,35.0,7339.0301,49.0,19943.3475,60.0,1.032480e+05,1726.490689,209.686574,407.007092,1720.799465
9,灶消烹饪机,3.0,3220.8890,1.0,0.7540,2.0,58.8380,3.0,3.280481e+03,1073.629667,0.754000,29.419000,1093.493667
